# Questões 2 e 3 - Schema e Carregamento

Geração do `schema.sql` a partir dos CSVs (Questão 2) e carga dos dados no Postgres (Questão 3).

## Questão 2 - Geração do schema

Script: `src/generate_schema.py` (apenas bibliotecas padrão do Python 3, conforme exigido).

**Lógica de inferência de tipos:**
- Percorre o CSV inteiro (não uma amostra) para evitar erro em colunas majoritariamente vazias no início do arquivo
- Ordem de checagem: BOOLEAN → TIMESTAMP → DATE → INTEGER/BIGINT → NUMERIC → TEXT
- Números com zero à esquerda (ex: telefone, código de barras) são tratados como TEXT, mesmo sendo só dígitos, pois convertê-los para tipo numérico perderia o zero e corromperia o dado
- Números grandes demais para `INTEGER` (32 bits) mas que cabem em `BIGINT` (ex: CPF, CNPJ) são promovidos automaticamente

In [16]:
import sys
import subprocess

result = subprocess.run([sys.executable, '-m', 'src.generate_schema'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("ERRO:", result.stderr)

Processado: addresses
Processado: attributes
Processado: brands
Processado: categories
Processado: customers
Processado: employees
Processado: fiscal_invoices
Processado: goods_receipt_items
Processado: goods_receipts
Processado: locations
Processado: order_items
Processado: orders
Processado: payments
Processado: product_suppliers
Processado: product_variants
Processado: products
Processado: purchase_order_items
Processado: purchase_orders
Processado: return_items
Processado: returns
Processado: stock_levels
Processado: stock_movements
Processado: suppliers
Processado: variant_attribute_values

Schema gerado em: sql\schema.sql



Schema gerado em `sql/schema.sql`. Para a carga inicial, o arquivo deve ser aplicado em um
banco vazio antes da execução de `src/load_csvs.py`.

**Nota de depuração:** na primeira versão do script, `tax_id`, `cpf`, `phone` e `barcode_ean`
foram inferidos como `INTEGER`, o que quebrou a carga (Questão 3) com erro `value out of range
for type integer`. Esses campos têm mais dígitos do que os 32 bits do `INTEGER` suportam
(ex: CNPJ com 14 dígitos). A correção acima (promoção para `BIGINT` ou `TEXT` quando há zero
à esquerda) resolveu o problema. Comando para aplicar o schema no terminal:

```powershell
Get-Content sql/schema.sql | docker exec -i lh_nautical_db psql -U lh_user -d lh_nautical
```

In [17]:
from src.db import get_engine
import pandas as pd

engine = get_engine()

tabelas = pd.read_sql(
    "SELECT table_name FROM information_schema.tables WHERE table_schema = 'public' ORDER BY table_name;",
    engine
)
print(f"Total de tabelas criadas: {len(tabelas)}")
tabelas

Total de tabelas criadas: 24


,table_name
0,addresses
1,attributes
2,brands
3,categories
4,customers
5,employees
6,fiscal_invoices
7,goods_receipt_items
8,goods_receipts
9,locations


## Questão 3 - Carregamento dos CSVs

Script: `src/load_csvs.py`. Usa `COPY` via `psycopg2` (carga em massa, sem tratamento de
nulos ou caracteres especiais; célula vazia no CSV vira `NULL` automaticamente). Antes da
carga, o script verifica se as tabelas já possuem dados para evitar duplicação.

In [18]:
from src.db import get_psycopg2_connection

conn = get_psycopg2_connection()
try:
    with open("sql/schema.sql", encoding="utf-8") as f:
        schema_sql = f.read()

    with conn.cursor() as cur:
        cur.execute(schema_sql)
    conn.commit()
finally:
    conn.close()

print("Schema aplicado com sucesso.")

Schema aplicado com sucesso.


In [19]:
result = subprocess.run([sys.executable, '-m', 'src.load_csvs'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("ERRO:", result.stderr)

Encontrados 24 arquivos CSV para carregar.

Carga nÃ£o executada: o banco jÃ¡ possui dados nas tabelas:
addresses, attributes, brands, categories, customers, employees, fiscal_invoices, goods_receipt_items, goods_receipts, locations, order_items, orders, payments, product_suppliers, product_variants, products, purchase_order_items, purchase_orders, return_items, returns, stock_levels, stock_movements, suppliers, variant_attribute_values



## Questão 3.2 - Validação

**Total de linhas somadas: customers, orders, order_items e payments**


In [20]:
validacao_query = """
SELECT
    (SELECT COUNT(*) FROM customers)    AS qtd_customers,
    (SELECT COUNT(*) FROM orders)       AS qtd_orders,
    (SELECT COUNT(*) FROM order_items)  AS qtd_order_items,
    (SELECT COUNT(*) FROM payments)     AS qtd_payments,
    (SELECT COUNT(*) FROM customers) + (SELECT COUNT(*) FROM orders)
        + (SELECT COUNT(*) FROM order_items) + (SELECT COUNT(*) FROM payments) AS total_linhas
"""

validacao = pd.read_sql(validacao_query, engine)
validacao

,qtd_customers,qtd_orders,qtd_order_items,qtd_payments,total_linhas
0,2000,48998,147320,53546,251864


**Resposta Q3.2: 251.864 linhas** (2.000 customers + 48.998 orders + 147.320 order_items + 53.546 payments)